In [2]:
from google.colab import drive
import pandas as pd

!pip install gurobipy
import gurobipy as gp
from gurobipy import GRB

import os

# Gurobi WLS credentials -- set these as environment variables, do not hardcode
params = {'WLSACCESSID': os.environ['GRB_WLSACCESSID'],
'WLSSECRET': os.environ['GRB_WLSSECRET'],
'LICENSEID': int(os.environ['GRB_LICENSEID'])
}

# Import file in Google Drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


DATA IMPORT, CLEANING, PREPARATION

In [4]:
#Historical data
df_actual_uncleaned = pd.read_excel(file_path, sheet_name='Pedidos 2024')
#Parameters for transport data
df_param = pd.read_excel(file_path, sheet_name='Vehículos')

NameError: name 'file_path' is not defined

In [ ]:
#CLEANING FOR PEDIDOS 2024

# Remove unnecessary columns
cols_to_remove = ['Base', 'Semana ', 'Equipo', 'Days (Fecha)', 'Mes']
df_actual_uncleaned = df_actual_uncleaned.drop(columns=cols_to_remove)
unnamed_cols = [col for col in df_actual_uncleaned.columns if 'Unnamed' in col]
df_actual_uncleaned = df_actual_uncleaned.drop(columns=unnamed_cols)
# Filter the DataFrame by Date
df_actual_uncleaned['Fecha'] = pd.to_datetime(df_actual_uncleaned['Fecha'])
start_date = '2024-12-09'
end_date = '2024-12-14'
df_actual = df_actual_uncleaned[(df_actual_uncleaned['Fecha'] >= start_date) & (df_actual_uncleaned['Fecha'] <= end_date)]
# Renaming Variables
new_column_names = {
    'Fecha': 'Date',
    'Origen': 'Origin',
    'EstadoZT': 'State_CL',
    'MunicipioZT': 'Municipality_CL',
    'PoblacionZT': 'County_CL',
    'Cliente': 'Client',
    'Vehiculo': 'Transport',
    'Viaje#': 'Trip_ID',
    ' Distancia': 'Distance',
    ' Lote': 'Batch',
}
df_actual = df_actual.rename(columns=new_column_names)

'''
#Define Trip_ID to Interger
df_actual['Trip_ID'] = pd.to_numeric(df_actual['Trip_ID'], errors='coerce').astype('Int64')

# Perform one-hot encoding on the 'Transport' column
df_actual = pd.get_dummies(df_actual, columns=['Transport'], prefix=['Transport'])
df_actual = df_actual.replace({False: 0, True: 1})
'''
df_demand = df_actual[['Date', 'County_CL', 'Client', 'Distance', 'Batch']].copy()
df_demand = df_demand.groupby(['Date', 'Client', 'County_CL','Distance'])['Batch'].sum().reset_index()
df_demand.insert(df_demand.columns.get_loc('Client'), 'Order', range(1, len(df_demand) + 1))

#CLEANING FOR PARAMETERS
# Renaming Variables
new_column_names = {
    'Vehiculo':'Transport',
    'Flota disponible': 'Available Fleet',
    'Capacidad tonelaje (local, <30 km de distancia)':'Local Capacity',
    'Capacidad tonelaje (foráneo, >30 km de distancia)':'External Capacity',
    'Viajes disponibles por día':'Available Trips per day',
    'KM por día disponibles':'Available KM per day',
    'Costo fijo por día':'Fixed Cost per day',
    'Costo variable por KM (ida)':'Variable Cost per KM'}
df_param = df_param.rename(columns=new_column_names)

# Replace 'camioneta' with 'Truck' in the 'Transport' column of df_param
df_param['Transport'] = df_param['Transport'].replace('Camioneta', 'Truck')

# Include Subcontracted Transport
new_rows = []
for i in range(4):
  new_row = df_param.iloc[i].copy()
  new_row['Transport'] = f'SC_{new_row["Transport"]}'  # Add SC_ to Transport
  new_row['Available Fleet'] = 20
  new_row['Fixed Cost per day'] = int(new_row['Fixed Cost per day'] * 1.25)  # Increase by 25%
  new_row['Variable Cost per KM'] = int(new_row['Variable Cost per KM'] * 1.25)  # Increase by 25%
  new_rows.append(new_row)
new_df = pd.DataFrame(new_rows)
df_param = pd.concat([df_param, new_df], ignore_index=True)

# Fix Real Cargo Capacity
df_param['Local Capacity'] = round(df_param['Local Capacity']*1.015,1)
df_param['External Capacity'] = round(df_param['External Capacity']*1.015,1)

df_param.insert(df_param.columns.get_loc('Transport'), 'Transport Type', range(1, len(df_param) + 1))

In [ ]:
df_param

,Transport Type,Transport,Available Fleet,Local Capacity,External Capacity,Available Trips per day,Available KM per day,Fixed Cost per day,Variable Cost per KM
0,1,Truck,10,5.1,5.1,3,100,200,50
1,2,Rabon,5,10.1,8.1,2,100,500,40
2,3,Torton,5,20.3,16.2,2,300,1000,30
3,4,Trailer,1,36.5,36.5,1,300,2000,20
4,5,SC_Truck,20,5.1,5.1,3,100,250,62
5,6,SC_Rabon,20,10.1,8.1,2,100,625,50
6,7,SC_Torton,20,20.3,16.2,2,300,1250,37
7,8,SC_Trailer,20,36.5,36.5,1,300,2500,25


LINEAR MODELING

In [ ]:
for date in df_demand['Date'].unique():
  print(date)

2024-12-09 00:00:00
2024-12-10 00:00:00
2024-12-11 00:00:00
2024-12-12 00:00:00
2024-12-13 00:00:00
2024-12-14 00:00:00


In [ ]:
#for date in df_demand['Date'].unique():
df_daily_demand = df_demand[df_demand['Date'] == '2024-12-09']
# -------------------------------
# Sets
# -------------------------------
orders = df_daily_demand['Order'].tolist()            # Set I: Orders
vehicle_types = df_param['Transport Type'].tolist()  # Set J: Vehicle types (1 to 8)
fleet_units = list(range(1, 21))      # Set K: Available fleet units (1 to 20)
trips = [1, 2, 3]                     # Set N: Trips per day
# -------------------------------
# Parameters (with dummy values)
# -------------------------------
DIS = df_demand.set_index('Order')['Distance'].to_dict()  # in km; note: distance == 30 is local
DEM = df_demand.set_index('Order')['Batch'].to_dict()  # demand in tonnes
# For each vehicle type j, define:
FC = df_param.set_index('Transport Type')['Fixed Cost per day'].to_dict() #Fixed Cost
VC = df_param.set_index('Transport Type')['Variable Cost per KM'].to_dict() #Variable Cost per KM
AF = df_param.set_index('Transport Type')['Available Fleet'].to_dict() #Available Fleet
AK = df_param.set_index('Transport Type')['Available KM per day'].to_dict() #Available KM per day
AT = df_param.set_index('Transport Type')['Available Trips per day'].to_dict() #Available Trips per day
LC = df_param.set_index('Transport Type')['Local Capacity'].to_dict()  # Local capacity in tonnes
EC = df_param.set_index('Transport Type')['External Capacity'].to_dict()  # External capacity in tonnes
# -------------------------------
# Create Model
# -------------------------------
env = gp.Env(params=params)
model = gp.Model("CEMEX_Transportation",env=env)
# -------------------------------
# Decision Variables
# -------------------------------
# y[i,j,k,n] : binary variable for using vehicle type j, fleet unit k, for order i on trip n.
y = model.addVars(orders, vehicle_types, fleet_units, trips, vtype=GRB.BINARY, name="y")

# x[i,j,k,n] : continuous variable for tonnage assigned for order i using vehicle type j, fleet unit k on trip n.
x = model.addVars(orders, vehicle_types, fleet_units, trips, vtype=GRB.CONTINUOUS, lb=0, name="x")

# z[j,k] : binary variable indicating if vehicle fleet unit k of type j is used.
z = model.addVars(vehicle_types, fleet_units, vtype=GRB.BINARY, name="z")

# -------------------------------
# Objective Function
# -------------------------------
# Minimize fixed costs for each used vehicle plus round-trip variable costs.
# Note: 2*DIS is used to represent the round-trip distance.
model.setObjective(
    gp.quicksum(FC[j] * z[j,k] for j in vehicle_types for k in fleet_units) +
    gp.quicksum(2 * DIS[i] * VC[j] * y[i,j,k,n]  for i in orders for j in vehicle_types for k in fleet_units for n in trips),
    GRB.MINIMIZE
)
# -------------------------------
# Constraints
# -------------------------------
# 1. Demand Fulfillment Constraint: For each order i, total tonnage assigned equals the demand.
model.addConstrs(
    (gp.quicksum(x[i,j,k,n] for j in vehicle_types for k in fleet_units for n in trips) == DEM[i]
    for i in orders),
    name="DemandFulfillment"
)
# 2. Available Fleet Constraint: For each vehicle type j, the number of vehicles used does not exceed AF[j].
model.addConstrs(
    (gp.quicksum(z[j,k] for k in fleet_units) <= AF[j] for j in vehicle_types),
    name="FleetLimit"
)
# 3. Available Trips and Linking Constraint:
# For each vehicle type j and fleet unit k, the total trips assigned is at most AT[j] if the vehicle is used.
model.addConstrs(
    (gp.quicksum(y[i,j,k,n] for i in orders for n in trips) <= AT[j] * z[j,k]
    for j in vehicle_types for k in fleet_units),
    name="TripLinking"
)
# 4. Available Kilometers Constraint: For each vehicle type j and fleet unit k,
# the total round-trip kilometers does not exceed AK[j].
model.addConstrs(
    (gp.quicksum(2 * y[i,j,k,n] * DIS[i] for i in orders for n in trips) <= AK[j]
    for j in vehicle_types for k in fleet_units),
    name="KmLimit"
)
# 5. Local and External Capacity Constraints & Linking x and y
# For each order i, vehicle type j, fleet unit k, and trip n:
for i in orders:
    for j in vehicle_types:
        for k in fleet_units:
            for n in trips:
                # If the trip is local (distance <= 30 km)
                if DIS[i] <= 30:
                    # Limit cargo to local capacity
                    model.addConstr(x[i,j,k,n] <= LC[j], name=f"LocalCapacity_i{i}_j{j}_k{k}_n{n}")
                    # Linking: if y is 0, then x must be 0. Here we use LC[j] as the big-M.
                    model.addConstr(x[i,j,k,n] <= LC[j] * y[i,j,k,n], name=f"LinkLocal_i{i}_j{j}_k{k}_n{n}")
                else:
                    # External trip: Limit cargo to external capacity
                    model.addConstr(x[i,j,k,n] <= EC[j], name=f"ExternalCapacity_i{i}_j{j}_k{k}_n{n}")
                    # Linking: if y is 0, then x must be 0. Use EC[j] as the big-M.
                    model.addConstr(x[i,j,k,n] <= EC[j] * y[i,j,k,n], name=f"LinkExternal_i{i}_j{j}_k{k}_n{n}")
# -------------------------------
# Optimize Model
# -------------------------------
model.optimize()
# -------------------------------
# Print Results
# -------------------------------
if model.status == GRB.OPTIMAL:
    print("\nOptimal solution found:")
    for v in model.getVars():
        if v.X > 0:
            print(f"{v.VarName}: {v.X}")
else:
    print("No optimal solution found.")


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value [REDACTED]
Academic license [REDACTED] - for non-commercial use only - registered to a0___@tec.mx
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Academic license [REDACTED] - for non-commercial use only - registered to a0___@tec.mx
Optimize a model with 9938 rows, 9760 columns and 28960 nonzeros
Model fingerprint: 0xcf1b49dd
Variable types: 4800 continuous, 4960 integer (4960 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+02]
  Objective range  [2e+02, 2e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+02]
Presolve removed 4808 rows and 1600 columns
Presolve time: 0.01s

Explored 0 nodes (0 simplex iterations) in 0.03 seconds (0.01 work units)
Thread count was 1 (of 2 available process